In [8]:
!pip install -q gradio nltk scikit-learn

In [9]:
# CodeAlpha Internship
# Artificial Intelligence
# Task 2: Chatbot for FAQs

import re
import nltk
import gradio as gr
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("stopwords")

faqs = [
    ("What is CodeAlpha?", "CodeAlpha is a software development company."),
    ("How many tasks are required?", "Complete at least two or three tasks."),
    ("Where do I upload my code?", "Upload your code to GitHub."),
    ("What should the repository be named?", "Use CodeAlpha_ProjectName."),
    ("Do I need to post on LinkedIn?", "Yes, post the project video on LinkedIn."),
    ("How do I submit my task?", "Submit it through the CodeAlpha form."),
    ("What certificates are provided?", "Completion and unique ID certificates are provided."),
    ("Can I get a recommendation letter?", "Yes, based on your performance."),
    ("Does CodeAlpha provide placement support?", "Yes, placement support is provided.")
]

stop_words = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

questions = [faq[0] for faq in faqs]
answers = [faq[1] for faq in faqs]

processed_questions = [preprocess(q) for q in questions]

vectorizer = TfidfVectorizer()
question_vectors = vectorizer.fit_transform(processed_questions)

def chatbot(message):
    if not message.strip():
        return "Please enter a question."

    processed_message = preprocess(message)
    message_vector = vectorizer.transform([processed_message])

    similarities = cosine_similarity(
        message_vector,
        question_vectors
    )[0]

    best_match = similarities.argmax()

    if similarities[best_match] < 0.15:
        return "Sorry, I don't know the answer."

    return answers[best_match]

with gr.Blocks(title="CodeAlpha Task 2") as app:

    gr.Markdown("# CodeAlpha Task 2: FAQ Chatbot")
    gr.Markdown("Ask questions about the CodeAlpha internship.")

    chatbot_ui = gr.Chatbot()
    message = gr.Textbox(
        label="Your Question",
        placeholder="Ask a question..."
    )

    send = gr.Button("Send")

    def respond(message, history):
        answer = chatbot(message)
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": answer})
        return "", history

    send.click(
        respond,
        inputs=[message, chatbot_ui],
        outputs=[message, chatbot_ui]
    )

app.launch(share=True)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91c52cb41261cefc03.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
